# ALVERIS: Copernicus DEM, InSAR Subsidence & Climate Value-at-Risk Engine
### Enterprise Earth Observation & Regulatory Financial Underwriting Pipeline
**Target Standards:** IPCC AR6 WG1 Chapter 9 · Basel III Pillar 3 Climate Disclosures · SEC Physical Climate Risk Rules · UN SDGs (11, 13, 15)  
**Core Technologies:** `GeoPandas` · `Rasterio` · `Xarray` · `Rioxarray` · `Shapely` · `Folium` · `SciPy` · `PyTorch`

---

## Executive Architecture Overview

Conventional commercial real estate appraisal relies on backward-looking 3-year sales comparables and static 3-band RGB satellite photography. This leaves institutional capital and lenders exposed to unpriced physical climate risks:
1. **Sub-centimeter Land Subsidence:** Groundwater depletion causing structural foundation shear.
2. **False Hydrologic Modeling ("Bathtub Fallacy"):** Slicing a planar elevation buffer without checking morphological connectivity to water bodies.
3. **RGB Blindness:** Inability to detect sub-surface soil salinization and root-zone moisture deficit before visible wilting.
4. **Unhedged Balance Sheets:** Physical hazards disconnected from discounted cash flow (DCF) haircuts and regulatory loan-to-value (LTV) limits.

```
+--------------------------------------------------------------------------------------------------+
|                                     ALVERIS CORE PIPELINE                                        |
+--------------------------------------------------------------------------------------------------+
|  [Cadastral GeoJSON] -> [GeoPandas Reprojection Engine: Dynamic Local UTM Zone]                  |
|                                |                                                                 |
|                                v                                                                 |
|  [Copernicus GLO-30 DEM]       [Sentinel-1 InSAR Multi-Decadal]     [Sentinel-2 13-Band L2A BOA] |
|  Rasterio Windowed Clip        Differential Settlement Field        Xarray / Rioxarray Data Cube |
|                                |                                                                 |
|                                v                                                                 |
|  [8-Way Morphological Flood] -> [Physical Risk Synthesis (0-100)]   [CNN Zoning Verification]    |
|                                |                                                                 |
|                                v                                                                 |
|  [Basel III Climate VaR Waterfall: DCF Haircuts, Remediation CapEx, Access Isolation Penalty]   |
|                                |                                                                 |
|                                v                                                                 |
|  [Interactive Folium GIS Map & Underwriting Memo Export]                                         |
+--------------------------------------------------------------------------------------------------+
```


## Step 1: Environment Setup & Core Geospatial Imports

We explicitly import industry-standard geospatial vector and raster libraries:
- `geopandas` for vector geometry parsing, coordinate reference system (CRS) transformations, and spatial indexing.
- `rasterio` for memory-efficient windowed reading, masking, and affine geotransforms.
- `rioxarray` and `xarray` for multi-dimensional spatiotemporal raster cubes.
- `folium` for interactive cartographic mapping and choropleth inspection.


In [1]:
import os
import sys
from pathlib import Path

# Add src directory to PYTHONPATH for local development
if "src" not in sys.path:
    sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import generate_binary_structure, label

# Core Vector GIS Stack
import geopandas as gpd
from shapely.geometry import Polygon, shape, box
import pyproj

# Core Raster GIS Stack
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
from rasterio.transform import Affine

# Multi-Dimensional Raster Cube Stack
import xarray as xr
import rioxarray

# Interactive GIS Mapping Stack
import folium
from folium import plugins

print("✅ All geospatial dependencies imported successfully!")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Rasterio:  {rasterio.__version__}")
print(f"  Xarray:    {xr.__version__}")
print(f"  Folium:    {folium.__version__}")


✅ All geospatial dependencies imported successfully!
  GeoPandas: 1.1.4
  Rasterio:  1.5.1
  Xarray:    2026.9.0
  Folium:    0.20.0


## Step 2: Cadastral Parcel Vector Ingestion & Geodetic Code Gates

Spatial calculations must maintain geodetic integrity:
- **Gate 1 (CRS Verification):** Unprojected geographic coordinates (`EPSG:4326`) express positions in angular degrees. Distance and area computed on degrees are physically invalid.
- **Dynamic UTM Determination:** Compute the parcel geographic centroid and automatically assign the correct metric UTM projection:
  $$\text{Zone} = \lfloor (\text{Lon} + 180) / 6 \rfloor + 1$$
  $$\text{Northern Hemisphere} \to \text{EPSG:326xx}, \quad \text{Southern Hemisphere} \to \text{EPSG:327xx}$$
- **Gate 4 (Area Plausibility):** Validate metric area ($m^2$ and hectares) against cadastral deed bounds.


In [2]:
# 1. Ingest Cadastral Parcel GeoJSON via GeoPandas
parcel_path = Path("../data/sample/coastal_periurban_parcel.geojson")
gdf_raw = gpd.read_file(parcel_path)

print(f"Cadastral Record Loaded: {gdf_raw.iloc[0]['name']}")
print(f"Source Coordinate Reference System: {gdf_raw.crs}")
print(f"Asset ID: {gdf_raw.iloc[0]['asset_id']}")

# 2. Derive Dynamic Metric UTM Zone from Centroid
cen_deg = gdf_raw.geometry.iloc[0].centroid
lon, lat = float(cen_deg.x), float(cen_deg.y)

utm_zone = int((lon + 180.0) // 6.0) + 1
utm_epsg = f"EPSG:{32600 + utm_zone if lat >= 0 else 32700 + utm_zone}"
print(f"\nGeographic Centroid: ({lat:.5f}°N, {lon:.5f}°E)")
print(f"Derived Projected Metric CRS: {utm_epsg}")

# 3. Reproject to UTM Metric Planar Coordinates
gdf_utm = gdf_raw.to_crs(utm_epsg)
parcel_geom_utm = gdf_utm.geometry.iloc[0]

# 4. Compute Planar Metrics
area_sqm = float(parcel_geom_utm.area)
area_ha = area_sqm / 10000.0
perimeter_m = float(parcel_geom_utm.length)

print(f"Planar Metric Area: {area_sqm:,.2f} m² ({area_ha:.2f} hectares)")
print(f"Cadastral Perimeter: {perimeter_m:,.2f} meters")

# Gate 4 Check: Prevent degrees-as-meters calculation
assert area_sqm > 50.0, "Area is suspiciously small! Check if coordinates were degrees."
print("✅ Spatial Gate 4 Passed: Area strictly positive and metric.")


Cadastral Record Loaded: Ennore Coastal Estuary Land Parcel
Source Coordinate Reference System: EPSG:4326
Asset ID: ALV-TN-CP01

Geographic Centroid: (13.24650°N, 80.31400°E)
Derived Projected Metric CRS: EPSG:32644
Planar Metric Area: 107,838.38 m² (10.78 hectares)
Cadastral Perimeter: 1,313.62 meters
✅ Spatial Gate 4 Passed: Area strictly positive and metric.


## Step 3: Copernicus GLO-30 DEM Processing via Rasterio

We perform windowed extraction and polygon clipping using `rasterio`:
- Windowed spatial read to minimize memory footprint.
- Masking against the cadastral boundary geometry.
- Computation of Horn's slope algorithm:
  $$\text{Slope} = \arctan\left(\sqrt{\left(\frac{\partial z}{\partial x}\right)^2 + \left(\frac{\partial z}{\partial y}\right)^2}\right)$$


In [3]:
from alveris.sample_data import get_or_generate_dem_for_parcel

# Ensure DEM exists for the parcel
geom_geojson = gdf_raw.geometry.iloc[0].__geo_interface__
asset_id = gdf_raw.iloc[0]["asset_id"]
dem_file = get_or_generate_dem_for_parcel(geom_geojson, asset_id, output_dir="../data/sample")

# Open with Rasterio and mask to parcel geometry
with rasterio.open(dem_file) as src:
    print(f"DEM Driver: {src.driver} | Dimensions: {src.width}x{src.height} | Bands: {src.count}")
    print(f"DEM CRS: {src.crs} | Resolution: {src.res}")
    
    out_image, out_transform = mask(src, [geom_geojson], crop=True)
    out_meta = src.meta.copy()

dem_elevation = out_image[0]
valid_cells = dem_elevation[dem_elevation != src.nodata] if src.nodata else dem_elevation[~np.isnan(dem_elevation)]

elev_min = float(np.min(valid_cells))
elev_mean = float(np.mean(valid_cells))
elev_max = float(np.max(valid_cells))
elev_p10 = float(np.percentile(valid_cells, 10))
elev_p90 = float(np.percentile(valid_cells, 90))

print(f"\nCopernicus GLO-30 Elevation Summary:")
print(f"  Minimum: {elev_min:.2f} m")
print(f"  10th Percentile: {elev_p10:.2f} m")
print(f"  Mean:    {elev_mean:.2f} m")
print(f"  90th Percentile: {elev_p90:.2f} m")
print(f"  Maximum: {elev_max:.2f} m")
print(f"  Topographic Relief (Max - Min): {elev_max - elev_min:.2f} m")


DEM Driver: GTiff | Dimensions: 100x100 | Bands: 1
DEM CRS: EPSG:4326 | Resolution: (0.00042999999999992155, 0.0004299999999999926)

Copernicus GLO-30 Elevation Summary:
  Minimum: 1.77 m
  10th Percentile: 1.87 m
  Mean:    2.07 m
  90th Percentile: 2.25 m
  Maximum: 2.29 m
  Topographic Relief (Max - Min): 0.52 m


## Step 4: Multi-Dimensional Raster Cubes via Xarray & Rioxarray

Modern Earth Observation pipelines treat multi-temporal imagery as **Spatiotemporal Data Cubes** $(T, C, Y, X)$.  
Here we demonstrate `xarray` and `rioxarray` integration:
- Open multi-band rasters with coordinate metadata.
- Calculate spectral indices: Normalized Difference Vegetation Index (NDVI) and Normalized Difference Moisture Index (NDMI).
- Inspect geospatial coordinate arrays (latitude, longitude, UTM easting/northing).


In [4]:
from alveris.sample_data import generate_sentinel2_bands_fixture

# Generate synthetic Sentinel-2 Level-2A BOA reflectance bands
grid_shape = dem_elevation.shape
b4, b5, b8, b11 = generate_sentinel2_bands_fixture(grid_shape, condition="healthy")

# Construct Multi-Dimensional Xarray Dataset
y_coords = np.arange(grid_shape[0])
x_coords = np.arange(grid_shape[1])

ds_bands = xr.Dataset(
    data_vars={
        "elevation": (("y", "x"), dem_elevation),
        "B04_red": (("y", "x"), b4),
        "B05_red_edge": (("y", "x"), b5),
        "B08_nir": (("y", "x"), b8),
        "B11_swir": (("y", "x"), b11),
    },
    coords={"y": y_coords, "x": x_coords},
    attrs={"crs": utm_epsg, "sensor": "Sentinel-2 MSI", "processing_level": "Level-2A BOA"},
)

# Compute Normalized Difference Spectral Indices across Xarray Tensors
ndvi = (ds_bands["B08_nir"] - ds_bands["B04_red"]) / (ds_bands["B08_nir"] + ds_bands["B04_red"] + 1e-6)
ndmi = (ds_bands["B08_nir"] - ds_bands["B11_swir"]) / (ds_bands["B08_nir"] + ds_bands["B11_swir"] + 1e-6)

ds_bands["NDVI"] = ndvi
ds_bands["NDMI"] = ndmi

print("✅ Xarray Earth Observation Cube Constructed:")
print(f"Variables: {list(ds_bands.data_vars)}")
print(f"Dimensions: {dict(ds_bands.dims)}")
print(f"Mean Parcel Canopy NDVI (Vigor): {float(ndvi.mean()):.4f}")
print(f"Mean Parcel Canopy NDMI (Moisture): {float(ndmi.mean()):.4f}")


✅ Xarray Earth Observation Cube Constructed:
Variables: ['elevation', 'B04_red', 'B05_red_edge', 'B08_nir', 'B11_swir', 'NDVI', 'NDMI']
Dimensions: {'y': 8, 'x': 8}
Mean Parcel Canopy NDVI (Vigor): 0.7714
Mean Parcel Canopy NDMI (Moisture): 0.5037


C:\Users\pravi\AppData\Local\Temp\ipykernel_15756\4081821433.py:32: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Dimensions: {dict(ds_bands.dims)}")


## Step 5: Multi-Decadal InSAR Subsidence Trajectory Modeling

Ground deformation is measured via interferometric synthetic aperture radar (InSAR):
- Annual Line-of-Sight (LOS) velocity in $mm/year$.
- Multi-decadal cumulative subsidence projection for loan underwriting:
  $$S(t) = v \cdot \Delta t + \frac{1}{2} \alpha (\Delta t)^2$$
- Structural Differential Settlement Gradient:
  $$\delta = \frac{v_{\max} - v_{\min}}{\text{Parcel Span}} \quad (\text{mm/m})$$
  Geotechnical standards classify $\delta > 0.50\,\text{mm/m}$ as severe structural hazard.


In [5]:
from alveris.sample_data import generate_subsidence_grid_fixture
from alveris.subsidence.engine import SubsidenceModelConfig, analyze_parcel_subsidence

sub_grid = generate_subsidence_grid_fixture(dem_elevation.shape, base_rate_mm_yr=9.5)
sub_config = SubsidenceModelConfig(base_year=2025, model_type="linear")

parcel_mask = ~np.isnan(dem_elevation)
sub_metrics = analyze_parcel_subsidence(
    subsidence_rate_grid_mm_yr=sub_grid,
    parcel_mask=parcel_mask,
    parcel_span_meters=float(perimeter_m / 4.0),
    config=sub_config,
)

print(f"InSAR Subsidence Analysis:")
print(f"  Mean Sinking Rate: {sub_metrics.mean_subsidence_rate_mm_year:.2f} mm/year")
print(f"  Rate Range: [{sub_metrics.min_subsidence_rate_mm_year:.2f}, {sub_metrics.max_subsidence_rate_mm_year:.2f}] mm/year")
print(f"  Differential Settlement Gradient: {sub_metrics.differential_gradient_mm_per_m:.4f} mm/m")
print(f"  Structural Hazard Tier: {sub_metrics.settlement_risk.value.upper()}")
print(f"  2050 Cumulative Sinking (25 yrs): {sub_metrics.projection_2050.cumulative_subsidence_m * 100:.1f} cm")
print(f"  2100 Cumulative Sinking (75 yrs): {sub_metrics.projection_2100.cumulative_subsidence_m * 100:.1f} cm")


InSAR Subsidence Analysis:
  Mean Sinking Rate: 9.40 mm/year
  Rate Range: [4.24, 15.18] mm/year
  Differential Settlement Gradient: 0.0333 mm/m
  Structural Hazard Tier: NEGLIGIBLE
  2050 Cumulative Sinking (25 yrs): 23.5 cm
  2100 Cumulative Sinking (75 yrs): 70.5 cm


## Step 6: Physics-Grounded 8-Way Connected Flood Simulation

We avoid the naive planar "bathtub fill" by deploying an 8-neighbor Moore morphological connected components algorithm:
- Inland basins below sea level remain dry unless an uninterrupted, hydraulically contiguous flow path links them to open coastal water.
- Evaluates Sea Level Rise (SLR) + Storm Surge scenarios combined with local InSAR subsidence.


In [6]:
from alveris.sample_data import generate_ocean_seed_mask
from alveris.inundation.engine import ConnectedInundationEngine, ScenarioParameters

# Eastern coastal boundary where seawater enters
ocean_mask = generate_ocean_seed_mask(dem_elevation.shape, seed_boundary="east")

inund_engine = ConnectedInundationEngine(connectivity_mode="8_way")
scenario = ScenarioParameters(
    water_level_rise_m=2.15,
    pixel_area_sqm=900.0,
    scenario_id="slr_2_15_extreme_surge",
)

inund_result, conn_flood, depth_grid = inund_engine.evaluate_parcel_scenario(
    elevation_grid=dem_elevation,
    parcel_mask=parcel_mask,
    ocean_seed_mask=ocean_mask,
    params=scenario,
)

print(f"Hydrologic Inundation Telemetry:")
print(f"  Scenario: {inund_result.scenario_id} (+{inund_result.water_level_rise_m:.2f}m rise)")
print(f"  Flooded Area: {inund_result.flooded_area_sqm:,.0f} m² ({inund_result.flooded_percent:.1f}%)")
print(f"  Mean Water Depth: {inund_result.flood_depth_mean_m:.2f} m")
print(f"  Max Water Depth:  {inund_result.max_flood_depth_m:.2f} m")
print(f"  Protected Inland Low Pockets: {inund_result.unconnected_low_pocket_count}")


Hydrologic Inundation Telemetry:
  Scenario: slr_2_15_extreme_surge (+2.15m rise)
  Flooded Area: 45,000 m² (78.1%)
  Mean Water Depth: 5600.72 m
  Max Water Depth:  10001.15 m
  Protected Inland Low Pockets: 0


## Step 7: Basel III & SEC Climate Value-at-Risk (Climate VaR) Financial Engine

Under Basel III Pillar 3 climate risk guidelines, physical hazards are converted into quantitative capital deductions:
1. **Direct Land Loss Haircut:** Proportion of permanently submerged usable cadastral land.
2. **Geotechnical CapEx Reserve:** Deep underpinning remediation cost for differential foundation settlement:
   $$\text{CapEx} = \text{Area} \cdot \text{Cost/m}^2 \cdot \left(\frac{\text{Rate}}{10.0}\right)$$
3. **Road Network Isolation Discount:** Penalty if all access links exceed the 0.30m vehicle passability threshold.


In [7]:
from alveris.valuation.engine import (
    PhysicalHazardInputs,
    evaluate_climate_valuation,
)
from alveris.network.resilience import RoadNetworkResilienceEngine
from alveris.sample_data import generate_sample_road_network_fixture
from alveris.sensors.multispectral import MultispectralBands, extract_multispectral_stress

baseline_val_inr = float(gdf_raw.iloc[0]["baseline_market_value_inr"])

# Multispectral environmental stress using MultispectralBands container
spectral_bands = MultispectralBands(
    b4_red=b4,
    b5_red_edge=b5,
    b8_nir=b8,
    b11_swir1=b11,
)
env_metrics = extract_multispectral_stress(
    bands=spectral_bands,
    parcel_mask=parcel_mask,
)

# Road network passability test
net_engine = RoadNetworkResilienceEngine()
net_metrics = net_engine.evaluate_route_resilience(
    base_graph=generate_sample_road_network_fixture(),
    origin_node=0,
    destination_node=2,
    edge_flood_depths={(0, 1): 0.65, (1, 2): 0.65, (0, 3): 0.0, (3, 2): 0.0},
    scenario_id=scenario.scenario_id,
)

hazards = PhysicalHazardInputs(
    inundation=inund_result,
    subsidence=sub_metrics,
    environment=env_metrics,
    network=net_metrics,
)

valuation = evaluate_climate_valuation(
    baseline_market_value_inr=baseline_val_inr,
    hazards=hazards,
)

print(f"=== BASEL III CLIMATE VALUATION WATERFALL ===")
print(f"Baseline Unadjusted Market Value: ₹ {baseline_val_inr:,.0f}")
print(f"  (-) Inundation Land Loss Haircut: ₹ {valuation.deductions.inundation_loss_inr:,.0f}")
print(f"  (-) Geotechnical CapEx Reserve:  ₹ {valuation.deductions.subsidence_capex_reserve_inr:,.0f}")
print(f"  (-) Accessibility & Road Penalty:₹ {valuation.deductions.accessibility_penalty_inr:,.0f}")
print(f"  (-) Environmental Soil Discount: ₹ {valuation.deductions.environmental_discount_inr:,.0f}")
print(f"--------------------------------------------------")
print(f"Total Collateral Haircut:          ₹ {valuation.deductions.total_haircut_inr:,.0f} (-{valuation.deductions.total_haircut_percent:.1f}%)")
print(f"Climate-Adjusted Collateral Value: ₹ {valuation.climate_adjusted_value_inr:,.0f}")
print(f"Regulatory Climate VaR Proxy:      -{valuation.climate_var_percent:.1f}%")


=== BASEL III CLIMATE VALUATION WATERFALL ===
Baseline Unadjusted Market Value: ₹ 18,500,000
  (-) Inundation Land Loss Haircut: ₹ 14,453,125
  (-) Geotechnical CapEx Reserve:  ₹ 0
  (-) Accessibility & Road Penalty:₹ 1,470,750
  (-) Environmental Soil Discount: ₹ 0
--------------------------------------------------
Total Collateral Haircut:          ₹ 15,725,000 (-85.0%)
Climate-Adjusted Collateral Value: ₹ 2,775,000
Regulatory Climate VaR Proxy:      -85.0%


## Step 8: Interactive Multi-Layer Folium GIS Cartography

Finally, we assemble an institutional-grade interactive GIS dashboard using `folium`:
- DarkMatter, Esri Satellite, and OSM basemaps.
- Cadastral parcel boundary polygon via GeoPandas.
- 8-way hydrologic inundation hazard extent.
- Persistent scatterer (PS-InSAR) displacement rate stations.
- Access road passability vectors (emerald green vs red).


In [8]:
from alveris.ingestion.parcel import load_parcel_from_geojson
from alveris.reporting.folium_map import create_alveris_folium_map

parcel_asset = load_parcel_from_geojson(parcel_path)

folium_map = create_alveris_folium_map(
    parcel=parcel_asset,
    inundation=inund_result,
    subsidence_rate_mm_yr=-float(sub_metrics.mean_subsidence_rate_mm_year),
    is_network_severed=net_metrics.is_physically_isolated,
    zoom_start=15,
)

# Render interactive map directly inside the notebook
folium_map
